# RecXML Pipeline

This notebook documents the original workflow. Outputs and execution counts have been removed. The reusable implementation is organized under `src/recxml/`.

In [ ]:
import pandas as pd
import numpy as np

<h2> Reading Data </h2>

In [ ]:
data = pd.read_table('MINDsmall_train\\behaviors.tsv', header=None, names=["ImpressionID", "UserID", "Time", "History", "Impressions"])
data.head()

In [ ]:
#Selecting 20% training data only due to resource constraints
total_rows = data.shape[0]
selected_rows = int(total_rows * 0.01)

print('Total rows: ' + str(total_rows))
data = data.sample(n=selected_rows, random_state=42)
print('After selecting 20% rows: ' + str(data.shape[0]))

In [ ]:
data = data.drop('Time', axis=1)
data = data.drop('ImpressionID', axis=1)
data.head()

<h2> Creating User Attribute Vectors </h2>

In [ ]:
# Creating user attribute vector
attributes = pd.DataFrame(data[['UserID', 'History']])
news_categories = pd.read_table('MINDsmall_train\\news.tsv', header=None)
news_categories = news_categories[[0, 1, 2]]
news_categories = news_categories.rename(columns={0: "NewsID", 1: "Category", 2: "Subcategory"})
news_categories.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
attributes.dropna(subset=['History'], inplace=True)
attributes['History'] = attributes['History'].astype(str)
attributes['News_interacted'] = attributes['History'].apply(lambda x: x.split())
attributes.head()

In [ ]:
news_mapping = {}
for index, row in news_categories.iterrows():
    news_id = row['NewsID']
    category = row['Category']
    subcategory = row['Subcategory']
    news_mapping[news_id] = {'Category': category, 'Subcategory': subcategory}

# Initialize an empty dictionary to store user attribute vectors
_user_attributes = {}

# Loop through each row in the first dataframe
for index, row in attributes.iterrows():
    user_id = row['UserID']
    news_ids = row['News_interacted']

    # Initialize an empty dictionary to store category and subcategory counts
    category_count = {}
    subcategory_count = {}

    # Loop through the news IDs that the user interacted with
    for news_id in news_ids:
        news_id = news_id.strip()
        news_row = news_categories[news_categories['NewsID'] == news_id].iloc[0]
        category = news_row['Category']
        subcategory = news_row['Subcategory']

        # Update category and subcategory counts
        category_count[category] = category_count.get(category, 0) + 1
        subcategory_count[subcategory] = subcategory_count.get(subcategory, 0) + 1

    # Create an attribute vector for the user
    attribute_vector = {
        **category_count,
        **subcategory_count
    }

    _user_attributes[user_id] = attribute_vector

# Create a new dataframe from the user attributes dictionary
user_attributes = pd.DataFrame.from_dict(_user_attributes, orient='index')

# Fill NaN values with 0
user_attributes = user_attributes.fillna(0)

In [ ]:
user_attributes = pd.read_pickle('user_attributes.pkl')

In [ ]:
user_attributes.index.name = 'UserID'
user_attributes.reset_index(inplace=True)

In [ ]:
user_attributes.head()

<h6>Checkpoint 1:</h6> Saving user attributes dataframe

In [ ]:
# Saving user_attributes vectors
user_attributes.to_pickle('user_attributes.pkl')

<h2>Preprocessing Data and Its Construction</h2>

Creating user-news interaction dataframe

In [ ]:

new_rows = []
data['Impressions'] = data['Impressions'].astype(str)

for index, row in data.iterrows():
    user_id = row['UserID']
    impression_news_ids = row['Impressions'].split(' ')
    # Iterate over each news id and create a new row
    for impression_news_id in impression_news_ids:
         new_rows.append({'UserID': user_id, 'NewsID': impression_news_id})

# Create a new dataframe from the new rows
data = pd.DataFrame(new_rows)

In [ ]:
data.head()

In [ ]:
data['Interaction'] = 1
data.loc[data['NewsID'].str.endswith("-0"), 'Interaction'] = -1
data['NewsID'] = data['NewsID'].str.rstrip("-0")
data['NewsID'] = data['NewsID'].str.rstrip("-1")

In [ ]:
data.head()

<h6>Checkpoint 2:</h6> Saving user-news interaction dataframe

In [ ]:
data.to_pickle('user-news-interaction.pkl')

<h2>Creating Unique Numeric IDs for Users and News </h2>

In [ ]:
# Creating UserID and NewsID to numeric UserID and NewsID mappings

unique_news_ids = data['NewsID'].unique()
unique_user_ids = data['UserID'].unique()

news_id_mapping = {news_id: i for i, news_id in enumerate(unique_news_ids)}
user_id_mapping = {user_id: i for i, user_id in enumerate(unique_user_ids)}

# Add new columns with the numeric index values
data['NewsID_Numeric'] = data['NewsID'].map(news_id_mapping)
data['UserID_Numeric'] = data['UserID'].map(user_id_mapping)

In [ ]:
# Create reverse mappings to map numeric IDs back to original IDs
reverse_news_id_mapping = {i: news_id for news_id, i in news_id_mapping.items()}
reverse_user_id_mapping = {i: user_id for user_id, i in user_id_mapping.items()}

In [ ]:
data.head()

<h6>Checkpoint 3:</h6> Saving user and news ID mappings and reverse mappings

In [ ]:
import pickle

In [ ]:
# Saving news mappings to pickle file
with open('news_id_mapping.pkl', 'wb') as file:
    pickle.dump(news_id_mapping, file)

##### Use the following lines to load news mappings if needed #####
# with open('news_id_mapping.pkl', 'rb') as file:
#     loaded_news_id_mapping = pickle.load(file)

# Saving user mappings to pickle file
with open('user_id_mapping.pkl', 'wb') as file:
    pickle.dump(user_id_mapping, file)

##### Use the following lines to load news mappings if needed #####
# with open('user_id_mapping.pkl', 'rb') as file:
#     loaded_user_id_mapping = pickle.load(file)

In [ ]:
# Saving reverse news mappings to pickle file
with open('reverse_news_id_mapping.pkl', 'wb') as file:
    pickle.dump(reverse_news_id_mapping, file)

##### Use the following lines to load reverse news mappings if needed #####
# with open('reverse_news_id_mapping.pkl', 'rb') as file:
#     loaded_reverse_news_id_mapping = pickle.load(file)

# Saving reverse user mappings to pickle file
with open('reverse_user_id_mapping.pkl', 'wb') as file:
    pickle.dump(reverse_user_id_mapping, file)

##### Use the following lines to load reverse user mappings if needed #####
# with open('reverse_user_id_mapping.pkl', 'rb') as file:
#     loaded_reverse_user_id_mapping = pickle.load(file)

Utility functions to map and reverse map original and numeric IDs

In [ ]:
def reverse_map_user_id(user_id_numeric, reverse_user_id_mapping):
    return reverse_user_id_mapping.get(user_id_numeric)

In [ ]:
def reverse_map_news_id(news_id_numeric, reverse_news_id_mapping):
    return reverse_news_id_mapping.get(news_id_numeric)

In [ ]:
def map_user_id(user_id, user_id_mapping):
    return user_id_mapping.get(user_id)

In [ ]:
def map_news_id(news_id, news_id_mapping):
    return news_id_mapping.get(news_id)

<h2>Adding Numeric UserID to User Attribute Vectors</h2>

In [ ]:
user_attributes['UserID_Numeric'] = user_attributes['UserID'].apply(lambda x: map_user_id(x, user_id_mapping))

In [ ]:
reordered_columns = ['UserID_Numeric'] + [col for col in user_attributes.columns if col != 'UserID_Numeric']
user_attributes = user_attributes[reordered_columns]

In [ ]:
user_attributes.head()

<h6>Checkpoint 4: </h6>Saving User Attributes Vector along with Numeric User IDs

In [ ]:
# Saving user_attributes vectors
user_attributes.to_pickle('user_attributes.pkl')

<h2>Preparing Training Data</h2>

In [ ]:
data = data.drop(['UserID', 'NewsID'], axis=1)
data = data[['NewsID_Numeric', 'UserID_Numeric', 'Interaction']]

In [ ]:
data.head()

<h6>Checkpoint 5:</h6> Saving cleaned interactions data numeric IDs and interactions

In [ ]:
data.to_pickle('user-news-interaction-cleaned.pkl')

Creating array of training data. Each row represents a unique news and each columns represents a unique user

In [ ]:
# Get unique user_ids and news_ids
user_ids = data['UserID_Numeric'].unique()
news_ids = data['NewsID_Numeric'].unique()

# Create an empty array with the appropriate dimensions
array = np.zeros((len(news_ids), len(user_ids)))

# Iterate over the DataFrame and populate the array
for _, row in data.iterrows():
    user_index = np.where(user_ids == row['UserID_Numeric'])[0][0]
    news_index = np.where(news_ids == row['NewsID_Numeric'])[0][0]
    array[news_index, user_index] = row['Interaction']

In [ ]:
print(array)
print('\nTraining Data Shape: ' + str(array.shape))

<h6>Checkpoint 6: </h6> Saving Training Data

In [ ]:
np.save('dataset.npy', array)

<h2>Creating Labels for News Items</h2>

Reading news data to create their embeddings

In [ ]:
# Reading news data in order to create their embeddings
news_data = pd.read_table('MINDsmall_train\\news.tsv', header=None, names=['NewsID', 'Category', 'Subcategory', 'Title', 'Abstract', 'URL', 'Title_Entities', 'Abstract_Entites'])

In [ ]:
news_data = news_data.drop('URL', axis=1)
news_data = news_data.drop('Title_Entities', axis=1)
news_data = news_data.drop('Abstract_Entites', axis=1)
news_data.head()

In [ ]:
# Reading news in order to keep only those news which have been interacted with
news_interacted = pd.read_pickle('user-news-interaction.pkl')
news_interacted.head()

In [ ]:
news_data = news_data[news_data['NewsID'].isin(news_interacted['NewsID'])]

Creating embeddings

In [ ]:
#from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from gensim.models import Word2Vec
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

Function to preprocess textual data in abstract and category etc.

In [ ]:
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenization
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(tokens)

Applying preprocessing to news data

In [ ]:
# Downloading wordnet for preprocessing
nltk.download('wordnet')

In [ ]:
news_data.dropna(subset=['Title', 'Abstract'], inplace=True)

In [ ]:
news_data['Processed_Text'] = news_data['Category'].apply(preprocess_text) + ' ' + news_data['Subcategory'].apply(preprocess_text) + ' ' + news_data['Title'].apply(preprocess_text) + ' ' + news_data['Abstract'].apply(preprocess_text)

In [ ]:
news_data.head()

Performing clustering to create and assign labels

In [ ]:
# Train Word2Vec model
model = Word2Vec(sentences=news_data['Processed_Text'], vector_size=100, window=5, min_count=1, workers=4)

In [ ]:
# Convert news texts to embeddings
def get_embeddings(words):
    vectors = [model.wv[word] for word in words if word in model.wv]
    if vectors:
        return sum(vectors) / len(vectors)
    else:
        return None

In [ ]:
news_data['Embeddings'] = news_data['Processed_Text'].apply(get_embeddings)
# Remove rows with no embeddings
news_data = news_data.dropna(subset=['Embeddings'])

In [ ]:
news_data.head()

In [ ]:
##### To be used on actual training data #####

# # Finding Optimal Number of Clusters using Silhouette Score
# max_clusters = 50  # Define a range of cluster numbers to try
# best_silhouette_score = -1
# best_num_clusters = 2

# for num_clusters in range(2, max_clusters + 1):
#     kmeans = KMeans(n_clusters=num_clusters, random_state=42)
#     kmeans.fit(list(news_data['Embeddings']))
#     cluster_labels = kmeans.labels_
#     silhouette_avg = silhouette_score(list(news_data['Embeddings']), cluster_labels)

#     if silhouette_avg > best_silhouette_score:
#         best_silhouette_score = silhouette_avg
#         best_num_clusters = num_clusters

# print("Best number of clusters:", best_num_clusters)

In [ ]:
best_num_clusters = 50 # No. of classes = 50 (using a fixed value for now)
kmeans = KMeans(n_clusters=best_num_clusters, random_state=42)
kmeans.fit(list(news_data['Embeddings']))
# Assign Labels
news_data['Label'] = kmeans.labels_

In [ ]:
news_data.head()

<h2>Saving Labels for Training</h2>

In [ ]:
import pickle
# loading news embeddings from a file
with open('news_id_mapping.pkl', 'rb') as file:
    news_id_mapping = pickle.load(file)
news_data['NewsID_Numeric'] = news_data['NewsID'].apply(lambda x: map_news_id(x, news_id_mapping))

In [ ]:
news_data.head()

<h6>Checkpoint 7:</h6> Saving news embeddings dataframe (with labels assigned)

In [ ]:
news_data.to_pickle('labelled_news_dataframe.pkl')

In [ ]:
news_data.shape

Creating labels array

In [ ]:
array_size = 4767 # Hard-coded value according to trainig data shape (train_data.shape[0])
default_label = 50 # can be changed later (according to no. of labels decided)
labels = np.full(array_size, default_label)

In [ ]:
labels[news_data['NewsID_Numeric']] = news_data['Label']

In [ ]:
print(labels)

<h6>Checkpoint 8: </h6>Saving labels array

In [ ]:
np.save('labels.npy', labels)